In [ ]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.8/930.8 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 97.0 MB/s eta 0:00:00


#T-statistic and P-value for IRQC

IMDB Top 250 Movies

In [ ]:
import pennylane as qml
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, mean_squared_error, mean_absolute_error

# Load and preprocess data
df = pd.read_csv('/content/IMDB Top 250 Movies.csv')
df.dropna(inplace=True)
user_movie_df = df.pivot_table("rating", "year", "name")

# Define a quantum circuit with 4 qubits
n_qubits = 5
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_corr(params):


    # Embed parameters using AngleEmbedding
    qml.templates.AngleEmbedding(params[:n_qubits], wires=range(n_qubits), rotation="X")

    # Apply Hadamard gates on each qubit
    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    # Apply RX and RY rotations
    for i in range(n_qubits):
        qml.RX(params[i], wires=i)
        qml.RY(params[i + n_qubits], wires=i)

    # Apply Controlled-Z gates between qubits
    for i in range(n_qubits - 1):
        qml.CZ(wires=[i, i + 1])

    # Apply entangling CNOT gates
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])

    # Measure the expectation of the PauliZ tensor product
    return qml.expval(qml.prod(*[qml.PauliZ(i) for i in range(n_qubits)]))

# Choose a movie for which to calculate correlations
movie_name = input("Enter movie name: ")
movie_name_df = user_movie_df[movie_name]

# Create a ground truth correlation matrix or rating matrix (this is just an example)
ground_truth_matrix = user_movie_df.corr().fillna(0)

# Calculate quantum correlations for each movie
movie_correlations = []
for column in user_movie_df.columns:
    if column != movie_name:
        params = np.random.uniform(0, 2 * np.pi, size=2 * n_qubits)
        correlation = quantum_corr(params)
        movie_correlations.append((column, correlation))

# Sort and display the top correlated movies
movie_correlations.sort(key=lambda x: x[1], reverse=True)
top_correlated = movie_correlations[:5]

print("\nTop correlated movies:")
for movie, correlation in top_correlated:
    print(f"{movie}: {correlation:.4f}")

# Extract the correlations from the top correlated movies
quantum_correlations = [correlation for movie, correlation in top_correlated]

# Extract the ground truth correlations from the ground truth matrix
ground_truth_correlations = [ground_truth_matrix.loc[movie_name, movie] for movie, _ in top_correlated]

# Calculate metrics
from sklearn.metrics import accuracy_score, precision_score, mean_absolute_error, mean_squared_error

#accuracy = accuracy_score(quantum_correlations, ground_truth_correlations)
#precision = precision_score(quantum_correlations, ground_truth_correlations, average='weighted')
mae = mean_absolute_error(quantum_correlations, ground_truth_correlations)
mse = mean_squared_error(quantum_correlations, ground_truth_correlations)
rmse = np.sqrt(mse)

#print(f"\nAccuracy: {accuracy:.4f}")
#print(f"Precision: {precision:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

Enter movie name: Ikiru

Top correlated movies:
Joker: 0.8381
Indiana Jones and the Last Crusade: 0.7353
Ratatouille: 0.7175
Metropolis: 0.6595
Star Wars: Episode V - The Empire Strikes Back: 0.6592
MAE: 0.7219
MSE: 0.5254
Root Mean Squared Error (RMSE): 0.7249


In [ ]:
from scipy.stats import ttest_rel

# Perform paired t-test
t_statistic, p_value = ttest_rel(quantum_correlations, ground_truth_correlations)

print(f"\nT-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference (p < 0.05)")
else:
    print("Result: No statistically significant difference (p ≥ 0.05)")


T-statistic: 22.0039
P-value: 0.0000
Result: Statistically significant difference (p < 0.05)


#supermarket sales

In [ ]:
import pennylane as qml
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load and preprocess data
df = pd.read_csv('/content/supermarket_sales - Sheet1.csv')
df.dropna(inplace=True)
user_product_df = df.pivot_table("Unit price", "Rating", "Product line")


# Define a quantum circuit with 5 qubits
n_qubits = 5
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_corr(params):

    # Embed parameters using AngleEmbedding
    qml.templates.AngleEmbedding(params[:n_qubits], wires=range(n_qubits), rotation="X")

    # Apply Hadamard gates on each qubit
    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    # Apply RX and RY rotations
    for i in range(n_qubits):
        qml.RX(params[i], wires=i)
        qml.RY(params[i + n_qubits], wires=i)

    # Apply Controlled-Z gates between qubits
    for i in range(n_qubits - 1):
        qml.CZ(wires=[i, i + 1])

    # Apply entangling CNOT gates
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])

    # Measure the expectation of the PauliZ tensor product
    return qml.expval(qml.prod(*[qml.PauliZ(i) for i in range(n_qubits)]))


# Choose a product for which to calculate correlations
product_name = input("Enter Product name: ")

if product_name not in user_product_df.columns:
    print("Error: Product name not found in dataset.")
    exit()

# Create a ground truth correlation matrix
ground_truth_matrix = user_product_df.corr().fillna(0)

# Calculate quantum correlations for each product
product_correlations = []
for column in user_product_df.columns:
    if column != product_name:
        # Generate random parameters for the quantum circuit
        params = np.random.uniform(0, 2 * np.pi, size=2 * n_qubits)
        correlation = quantum_corr(params)
        product_correlations.append((column, correlation))

# Sort and display the top correlated products
product_correlations.sort(key=lambda x: x[1], reverse=True)
top_correlated = product_correlations[:5]

print("\nTop correlated products:")
for product, correlation in top_correlated:
    print(f"{product}: {correlation:.4f}")

# Extract quantum correlations
quantum_correlations = [correlation for product, correlation in top_correlated]

# Extract the ground truth correlations from the ground truth matrix
ground_truth_correlations = [ground_truth_matrix.loc[product_name, product] for product, _ in top_correlated]

# Calculate error metrics
mae = mean_absolute_error(quantum_correlations, ground_truth_correlations)
mse = mean_squared_error(quantum_correlations, ground_truth_correlations)
rmse = np.sqrt(mse)

# Display error metrics
print(f"\nMean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")


Enter Product name: Health and beauty

Top correlated products:
Food and beverages: 0.4347
Home and lifestyle: 0.2141
Fashion accessories: 0.1428
Sports and travel: 0.0120
Electronic accessories: -0.0306

Mean Absolute Error (MAE): 0.1478
Mean Squared Error (MSE): 0.0346
Root Mean Squared Error (RMSE): 0.1861


In [ ]:
from scipy.stats import ttest_rel

# Perform paired t-test
t_statistic, p_value = ttest_rel(quantum_correlations, ground_truth_correlations)

print(f"\nT-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference (p < 0.05)")
else:
    print("Result: No statistically significant difference (p ≥ 0.05)")


T-statistic: 2.1369
P-value: 0.0994
Result: No statistically significant difference (p ≥ 0.05)


#BigBasket Products

In [ ]:
import pennylane as qml
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, mean_squared_error, mean_absolute_error

# Load and preprocess data
df = pd.read_csv('/content/BigBasket Products.csv')
df.dropna(inplace=True)
user_product_df = df.pivot_table("sale_price", "rating", "product")


# Define a quantum circuit with 5 qubits
n_qubits = 5
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_corr(params):

    # Embed parameters using AngleEmbedding
    qml.templates.AngleEmbedding(params[:n_qubits], wires=range(n_qubits), rotation="X")

    # Apply Hadamard gates on each qubit
    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    # Apply RX and RY rotations
    for i in range(n_qubits):
        qml.RX(params[i], wires=i)
        qml.RY(params[i + n_qubits], wires=i)

    # Apply Controlled-Z gates between qubits
    for i in range(n_qubits - 1):
        qml.CZ(wires=[i, i + 1])

    # Apply entangling CNOT gates
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])

    # Measure the expectation of the PauliZ tensor product
    return qml.expval(qml.prod(*[qml.PauliZ(i) for i in range(n_qubits)]))


# Choose a movie for which to calculate correlations
product_name = input("Enter Product name: ")
product_name_df = user_product_df[product_name]

# Create a ground truth correlation matrix or rating matrix (this is just an example)
ground_truth_matrix = user_product_df.corr().fillna(0)

# Calculate quantum correlations for each product
product_correlations = []
for column in user_product_df.columns:
    if column != product_name:
        # Generate random parameters for the quantum circuit
        params = np.random.uniform(0, 2 * np.pi, size=2 * n_qubits)
        correlation = quantum_corr(params)
        product_correlations.append((column, correlation))

# Sort and display the top correlated movies
product_correlations.sort(key=lambda x: x[1], reverse=True)
top_correlated = product_correlations[:5]

print("\nTop correlated products:")
for product, correlation in top_correlated:
    print(f"{product}: {correlation:.4f}")

# Extract the correlations from the top correlated movies
quantum_correlations = [correlation for product, correlation in top_correlated]

# Extract the ground truth correlations from the ground truth matrix
ground_truth_correlations = [ground_truth_matrix.loc[product_name, product] for product, _ in top_correlated]

# Calculate metrics
from sklearn.metrics import accuracy_score, precision_score, mean_absolute_error, mean_squared_error

mae = mean_absolute_error(quantum_correlations, ground_truth_correlations)
mse = mean_squared_error(quantum_correlations, ground_truth_correlations)
rmse = np.sqrt(mse)

print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")

Enter Product name: Tamarind/Hunisehannu

Top correlated products:
Rich Creme Hair Colour - Shade 4.16 Burgundy, Single Use: 0.9549
Classic Detergent - Lemon, Imported: 0.9457
Mix - Gulab Jamun: 0.9085
Nebula Multi Utility Vowel Plastic Roti Basket With Lid -Assorted Colour: 0.9083
Gelcrystal: 0.9054
MAE: 0.9245
MSE: 0.8552
RMSE: 0.9248


In [ ]:
from scipy.stats import ttest_rel

# Perform paired t-test
t_statistic, p_value = ttest_rel(quantum_correlations, ground_truth_correlations)

print(f"\nT-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference (p < 0.05)")
else:
    print("Result: No statistically significant difference (p ≥ 0.05)")


T-statistic: 87.0338
P-value: 0.0000
Result: Statistically significant difference (p < 0.05)


#MovieLens 10k

In [ ]:
import pennylane as qml
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, mean_squared_error, mean_absolute_error

# Load and preprocess data
movies = pd.read_csv('/content/movies.csv')
ratings = pd.read_csv('/content/ratings.csv')
df = movies.merge(ratings, how="left", on="movieId")
df.dropna(inplace=True)

user_movie_df = df.pivot_table("rating", "userId", "title")

# Define a quantum circuit with 5 qubits
n_qubits = 5
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_corr(params):
    # Apply Hadamard gates on each qubit
    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    # Apply RX and RY rotations
    for i in range(n_qubits):
        qml.RX(params[i], wires=i)
        qml.RY(params[i + n_qubits], wires=i)

    for i in range(n_qubits - 1):
        qml.CZ(wires=[i, i + 1])

    # Apply entangling CNOT gates
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])

    # Measure the expectation of the PauliZ tensor product on qubits
    return qml.expval(qml.prod(*[qml.PauliZ(i) for i in range(n_qubits)]))


# Choose a movie for which to calculate correlations
movie_name = input("Enter movie name: ")
movie_name_df = user_movie_df[movie_name]

# Create a ground truth correlation matrix or rating matrix (this is just an example)
ground_truth_matrix = user_movie_df.corr()

# Calculate quantum correlations for each movie
movie_correlations = []
for column in user_movie_df.columns:
    if column != movie_name:
        #params = np.random.uniform(0, 2 * np.pi, size=2)
        params = np.random.uniform(0, 2 * np.pi, size=2 * n_qubits)
        correlation = quantum_corr(params)
        movie_correlations.append((column, correlation))

# Sort and display the top correlated movies
movie_correlations.sort(key=lambda x: x[1], reverse=True)
top_correlated = movie_correlations[:5]

print("\nTop correlated movies:")
for movie, correlation in top_correlated:
    print(f"{movie}: {correlation:.4f}")

# Extract the correlations from the top correlated movies
quantum_correlations = [correlation for movie, correlation in top_correlated]

# Extract the ground truth correlations from the ground truth matrix
ground_truth_correlations = [
    ground_truth_matrix.loc[movie_name, movie]
    if pd.notna(ground_truth_matrix.loc[movie_name, movie]) else 0
    for movie, _ in top_correlated
]

# Ensure no NaNs in quantum_correlations or ground_truth_correlations
quantum_correlations = np.nan_to_num(quantum_correlations, nan=0.0)
ground_truth_correlations = np.nan_to_num(ground_truth_correlations, nan=0.0)

# Calculate metrics
mae = mean_absolute_error(quantum_correlations, ground_truth_correlations)
mse = mean_squared_error(quantum_correlations, ground_truth_correlations)
rmse = np.sqrt(mse)

print(f"\nMAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")

Enter movie name: Heat (1995)

Top correlated movies:
Little Nikita (1988): 0.9969
Romancing the Stone (1984): 0.9960
Creep 2 (2017): 0.9918
The Wild One (1953): 0.9883
The Boy and the Beast (2015): 0.9867

MAE: 0.9348
MSE: 0.8865
RMSE: 0.9415


In [ ]:
from scipy.stats import ttest_rel

# Perform paired t-test
t_statistic, p_value = ttest_rel(quantum_correlations, ground_truth_correlations)

print(f"\nT-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference (p < 0.05)")
else:
    print("Result: No statistically significant difference (p ≥ 0.05)")


T-statistic: 16.6372
P-value: 0.0001
Result: Statistically significant difference (p < 0.05)


#classical T-statistic and P-value:

#IMDB Top 250 Movies

In [ ]:
import pandas as pd
import numpy as np

# Read the CSV file
df = pd.read_csv('/content/IMDB Top 250 Movies.csv')

# Drop rows with missing values
df.dropna(inplace=True)

# Create a pivot table to get the average ratings
user_movie_df = df.pivot_table("year", "rating", "name")

user_movie_df.fillna(0, inplace=True)

#user_movie_df.fillna(0, inplace=True)
# Input the movie name
movie_name = input("Enter movie name: ")

# Select the ratings for the entered movie
movie_name_df = user_movie_df[movie_name]

# Calculate the correlation of the entered movie with all other movies
movie_corr = user_movie_df.corrwith(movie_name_df).sort_values(ascending=False)

movie_corr = movie_corr.dropna()

# Remove the input movie from the top correlated movies
movie_corr = movie_corr.drop(movie_name, errors='ignore')

# Select the top 5 correlated movies
top_5_corr_movies = movie_corr.head(5)
print(top_5_corr_movies)

# Calculate RMSE, MSE, and MAE
actual_ratings = user_movie_df[movie_name]
predicted_ratings = user_movie_df[top_5_corr_movies.index].mean(axis=1)
errors = actual_ratings - predicted_ratings
rmse = np.sqrt((errors ** 2).mean())
mse = (errors ** 2).mean()
mae = errors.abs().mean()

print(f"\nError Metrics for '{movie_name}':")
print(f"RMSE: {rmse:.2f}")
print(f"MSE: {mse:.2f}")
print(f"MAE: {mae:.2f}")

Enter movie name: Ikiru
name
A Separation                             1.0
Amélie                                   1.0
Dangal                                   1.0
Eternal Sunshine of the Spotless Mind    1.0
Citizen Kane                             1.0
dtype: float64

Error Metrics for 'Ikiru':
RMSE: 11.82
MSE: 139.60
MAE: 3.28


In [ ]:
from scipy.stats import ttest_rel

# Perform paired t-test
t_statistic, p_value = ttest_rel(predicted_ratings, actual_ratings)

print(f"\nT-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference (p < 0.05)")
else:
    print("Result: No statistically significant difference (p ≥ 0.05)")


T-statistic: 1.0000
P-value: 0.3370
Result: No statistically significant difference (p ≥ 0.05)


#MovieLens 10k

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load the datasets
movies = pd.read_csv('/content/movies.csv')
ratings = pd.read_csv('/content/ratings.csv')

# Merge movie titles with ratings
df = movies.merge(ratings, how="left", on="movieId")
df.dropna(inplace=True)

# Filter rare movies with fewer than 50 ratings
movie_counts = df["title"].value_counts()
rare_movies = movie_counts[movie_counts <= 50].index
common_movies = df[~df["title"].isin(rare_movies)]

# Train-test split based on userId
train_df, test_df = train_test_split(common_movies, test_size=0.2, random_state=42)

# Create pivot table (user-movie matrix) from training data
train_pivot = train_df.pivot_table(values="rating", index="userId", columns="title")

# Input movie name
movie_name = input("Enter movie name: ")

if movie_name not in train_pivot.columns:
    print(f"Movie '{movie_name}' not found in the training set.")
    exit()

# Calculate correlation of selected movie with all others in training data
target_ratings = train_pivot[movie_name]
correlations = train_pivot.corrwith(target_ratings)
correlations = correlations.dropna().sort_values(ascending=False)
correlations = correlations.drop(labels=movie_name, errors='ignore')
top_5_similar = correlations.head(5)

print(f"\nTop 5 movies similar to '{movie_name}':")
print(top_5_similar.to_string())

# Create dictionary of similar movie correlations
similar_movies = top_5_similar.index.tolist()
similar_weights = top_5_similar.values

# Predict ratings for the selected movie in the test set
test_subset = test_df[test_df["title"] == movie_name]

predictions = []
actuals = []

for idx, row in test_subset.iterrows():
    user = row["userId"]
    actual_rating = row["rating"]

    # Get ratings the user gave to the top 5 similar movies
    user_ratings = train_pivot.loc[user, similar_movies] if user in train_pivot.index else pd.Series(dtype=float)

    # Drop missing ratings
    valid_ratings = user_ratings.dropna()

    if len(valid_ratings) > 0:
        # Get corresponding weights for rated movies
        rated_movies = valid_ratings.index
        rated_weights = top_5_similar[rated_movies]

        # Weighted average prediction
        prediction = np.dot(valid_ratings.values, rated_weights.values) / rated_weights.values.sum()
        predictions.append(prediction)
        actuals.append(actual_rating)

# Evaluate prediction performance
if predictions:
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)

    print(f"\nError Metrics for predicted ratings of '{movie_name}':")
    print(f"RMSE: {rmse:.4f}")
    print(f"MSE : {mse:.4f}")
    print(f"MAE : {mae:.4f}")
else:
    print("Not enough data to compute error metrics (no matching users in test set).")


Enter movie name: Heat (1995)

Top 5 movies similar to 'Heat (1995)':
title
Slumdog Millionaire (2008)                       0.822222
Philadelphia (1993)                              0.802527
12 Angry Men (1957)                              0.749979
Aliens (1986)                                    0.706279
Harry Potter and the Half-Blood Prince (2009)    0.671984

Error Metrics for predicted ratings of 'Heat (1995)':
RMSE: 0.7544
MSE : 0.5692
MAE : 0.6255


In [ ]:
from scipy.stats import ttest_rel

# Perform paired t-test
t_statistic, p_value = ttest_rel(predictions, actuals)

print(f"\nT-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference (p < 0.05)")
else:
    print("Result: No statistically significant difference (p ≥ 0.05)")


T-statistic: -1.1058
P-value: 0.2947
Result: No statistically significant difference (p ≥ 0.05)


#BigBasket Products

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

df = pd.read_csv('/content/BigBasket Products.csv')


# We are deleting the missing values with a small number of missing values in the dataset
df.dropna(inplace=True)

# We are creating a table that includes the average ratings.
user_product_df = df.pivot_table("sale_price","rating", "product")

user_product_df.fillna(0, inplace=True)

# Split the data into training and testing sets
train, test = train_test_split(user_product_df, test_size=0.2, random_state=42)

# Select a movie for which you want to find similar movies
product_name = input("Enter product name: ")

if product_name in user_product_df.columns:
    # Extract ratings for the selected movie
    product_name_df = user_product_df[product_name]

    # Calculate the correlation of the selected movie with all other movies
    product_corr = user_product_df.corrwith(product_name_df).sort_values(ascending=False)
    product_corr = product_corr.dropna()

    # Remove the input movie from the top correlated movies
    product_corr = product_corr.drop(product_name, errors='ignore')

    # Print the top correlated movies
    top_correlated_products = product_corr.head(5)
    print("Top 5 products similar to '{}':".format(product_name))
    print(top_correlated_products)

actual_ratings = np.array([4.5, 4.0, 5.0, 3.5, 4.8])

# Calculate RMSE
rmse = np.sqrt(((top_correlated_products - actual_ratings) ** 2).mean())

# Calculate MSE
mse = ((top_correlated_products - actual_ratings) ** 2).mean()

# Calculate MAE
mae = (top_correlated_products - actual_ratings).abs().mean()

print(f"\nError Metrics for '{product_name}':")
print(f"RMSE: {rmse:.2f}")
print(f"MSE: {mse:.2f}")
print(f"MAE: {mae:.2f}")

Enter product name: Tamarind/Hunisehannu
Top 5 products similar to 'Tamarind/Hunisehannu':
product
After Shave Splash - Arctic Ice                                1.0
Aika Snack Bowl Set                                            1.0
Namkeen - Plain Boondi                                         1.0
Fruit Yoghurt - Peach                                          1.0
Airtight Plastic Container Set - Printed, Super Stylo, Pink    1.0
dtype: float64

Error Metrics for 'Tamarind/Hunisehannu':
RMSE: 3.40
MSE: 11.59
MAE: 3.36


In [ ]:
from scipy.stats import ttest_rel

# Perform paired t-test
t_statistic, p_value = ttest_rel(top_correlated_products, actual_ratings)

print(f"\nT-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference (p < 0.05)")
else:
    print("Result: No statistically significant difference (p ≥ 0.05)")


T-statistic: -12.3018
P-value: 0.0003
Result: Statistically significant difference (p < 0.05)


#supermarket sales

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

df = pd.read_csv('/content/supermarket_sales - Sheet1.csv')


# We are deleting the missing values with a small number of missing values in the dataset
df.dropna(inplace=True)

# We are creating a table that includes the average ratings.
user_product_df = df.pivot_table("Unit price","Rating", "Product line")


# Split the data into training and testing sets
train, test = train_test_split(user_product_df, test_size=0.2, random_state=42)

# Select a movie for which you want to find similar movies
product_name = input("Enter product name: ")

if product_name in user_product_df.columns:
    # Extract ratings for the selected movie
    product_name_df = user_product_df[product_name]

    # Calculate the correlation of the selected movie with all other movies
    product_corr = user_product_df.corrwith(product_name_df).sort_values(ascending=False)
    product_corr = product_corr.dropna()

    # Remove the input movie from the top correlated movies
    product_corr = product_corr.drop(product_name, errors='ignore')

    # Print the top correlated movies
    top_correlated_products = product_corr.head(5)
    print("Top 5 products similar to '{}':".format(product_name))
    print(top_correlated_products)


actual_ratings = np.array([4.5, 4.0, 5.0, 3.5, 4.8])

# Calculate RMSE
rmse = np.sqrt(((top_correlated_products - actual_ratings) ** 2).mean())

# Calculate MSE
mse = ((top_correlated_products - actual_ratings) ** 2).mean()

# Calculate MAE
mae = (top_correlated_products - actual_ratings).abs().mean()

print(f"\nError Metrics for '{product_name}':")
print(f"RMSE: {rmse}")
print(f"MSE: {mse}")
print(f"MAE: {mae}")


Enter product name: Health and beauty
Top 5 products similar to 'Health and beauty':
Product line
Food and beverages        0.147704
Sports and travel         0.030548
Electronic accessories   -0.019326
Home and lifestyle       -0.021837
Fashion accessories      -0.043420
dtype: float64

Error Metrics for 'Health and beauty':
RMSE: 4.376179041373829
MSE: 19.15094300215957
MAE: 4.341266147809121


In [ ]:
from scipy.stats import ttest_rel

# Perform paired t-test
t_statistic, p_value = ttest_rel(top_correlated_products, actual_ratings)

print(f"\nT-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant difference (p < 0.05)")
else:
    print("Result: No statistically significant difference (p ≥ 0.05)")


T-statistic: -15.7383
P-value: 0.0001
Result: Statistically significant difference (p < 0.05)


#end